# 비전 파이프라인 테스트

각 컴포넌트를 독립적으로 테스트합니다.

---
## 1. 흰 배경 분리 (Segmentation)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from pipeline.segmentation import segment_on_white, visualize_segmentation

# 테스트할 이미지 경로
IMG_PATH = "test_image.jpg"  # 흰 배경 위 분실물 사진으로 변경

img = cv2.imread(IMG_PATH)
if img is None:
    print(f"이미지 없음: {IMG_PATH}")
else:
    masked, mask, bbox = segment_on_white(img)
    vis = visualize_segmentation(img)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, im, title in zip(axes,
                              [img[:,:,::-1], mask, vis[:,:,::-1]],
                              ["원본", "마스크", "바운딩박스"]):
        ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
        ax.set_title(title)
        ax.axis("off")
    plt.suptitle(f"bbox: {bbox}", fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f"bbox: {bbox}")

### 임계값 튜닝
`s_thresh` (채도), `v_thresh` (밝기)를 조정해 분리 품질을 높입니다.

In [ ]:
# 슬라이더 없이 여러 임계값 비교
params = [(30, 200), (40, 180), (50, 170), (60, 160)]

if img is not None:
    fig, axes = plt.subplots(1, len(params), figsize=(5 * len(params), 4))
    for ax, (s, v) in zip(axes, params):
        _, mask_t, bbox_t = segment_on_white(img, s_thresh=s, v_thresh=v)
        ax.imshow(mask_t, cmap="gray")
        ax.set_title(f"s<{s}, v>{v}\nbbox={'있음' if bbox_t else '없음'}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

---
## 2. 색상 추출 (Color Extractor)

In [ ]:
from pipeline.color_extractor import extract_dominant_colors, color_name
import numpy as np

if img is not None:
    colors = extract_dominant_colors(img, mask, n_colors=3)
    print("주요 색상:")
    for c in colors:
        print(f"  {c['hex']}  {color_name(c['rgb']):<4}  비율: {c['ratio']:.1%}")

    # 색상 팔레트 시각화
    palette = np.zeros((80, 80 * len(colors), 3), dtype=np.uint8)
    for i, c in enumerate(colors):
        r, g, b = c["rgb"]
        palette[:, i*80:(i+1)*80] = [b, g, r]  # BGR

    plt.figure(figsize=(6, 1.5))
    plt.imshow(palette[:, :, ::-1])
    plt.axis("off")
    plt.title("추출된 색상 팔레트")
    plt.show()

---
## 3. 카메라 캘리브레이션

In [ ]:
from pipeline.calibration import calibrate_camera, save_calibration, load_calibration

# 체커보드 이미지 폴더 (직접 촬영 후 경로 지정)
CALIB_DIR = Path("calib_images")

if CALIB_DIR.exists():
    image_paths = list(CALIB_DIR.glob("*.jpg")) + list(CALIB_DIR.glob("*.png"))
    print(f"캘리브레이션 이미지: {len(image_paths)}장")

    calib = calibrate_camera(
        [str(p) for p in image_paths],
        board_size=(9, 6),    # 체커보드 내부 코너 수
        square_size_mm=25.0,  # 한 칸 크기 (mm)
    )
    save_calibration(calib, "camera_calib.json")

    import numpy as np
    print("\n카메라 행렬:")
    print(np.array(calib["camera_matrix"]))
    print(f"\n재투영 오차: {calib['reprojection_error']} px")
else:
    print(f"폴더 없음: {CALIB_DIR}")
    print("체커보드 이미지를 calib_images/ 폴더에 넣고 다시 실행하세요.")

---
## 4. Homography (픽셀 → Dobot 좌표)

In [ ]:
from pipeline.homography import compute_homography, pixel_to_dobot, reprojection_error, save_homography

# 실제 측정값으로 교체 필요
# 방법: Dobot을 특정 XY 위치로 이동 → 카메라에서 해당 위치의 픽셀 좌표 기록
pixel_pts = [
    (100, 100), (300, 100), (500, 100),
    (100, 300), (300, 300), (500, 300),
    (100, 500), (300, 500), (500, 500),
]
dobot_pts = [  # Dobot XY (mm), 실제 측정값으로 교체
    (200, 150), (180, 100), (160,  50),
    (220, 150), (200, 100), (180,  50),
    (240, 150), (220, 100), (200,  50),
]

H = compute_homography(pixel_pts, dobot_pts)
err = reprojection_error(pixel_pts, dobot_pts, H)
save_homography(H, "homography.json")

# 변환 테스트
test_px = (300, 300)
xyz = pixel_to_dobot(test_px, H, z=50.0)
print(f"\n픽셀 {test_px} → Dobot {xyz}")

---
## 5. 전체 파이프라인 통합

In [ ]:
from pipeline.pipeline import VisionPipeline
from pipeline.homography import load_homography
from pipeline.calibration import load_calibration
import cv2

# 모델 학습 완료 후 경로 지정
MODEL_PATH = "runs/full_run_v1/weights/best.pt"

# 필요한 파일이 있을 때만 실행
from pathlib import Path

missing = []
if not Path(MODEL_PATH).exists():
    missing.append(f"모델: {MODEL_PATH}")
if not Path("homography.json").exists():
    missing.append("homography.json (섹션 4 실행 필요)")

if missing:
    print("아직 준비 안 된 항목:")
    for m in missing:
        print(f"  - {m}")
else:
    H     = load_homography("homography.json")
    calib = load_calibration("camera_calib.json") if Path("camera_calib.json").exists() else None

    pipeline = VisionPipeline(
        model_path=MODEL_PATH,
        homography=H,
        calibration=calib,
        pickup_z=50.0,
    )

    img = cv2.imread(IMG_PATH)
    result = pipeline.run(img)
    print("파이프라인 결과:")
    for k, v in result.items():
        print(f"  {k}: {v}")